In [7]:
import json
import os
import urllib.error
import urllib.request

base_url = "https://api.subq.ai/v1"

request = urllib.request.Request(
    f"{base_url}/systemone",
    method="POST",
    headers={
        "Authorization": f"Bearer {os.environ['SUBQ_API_KEY']}",
        "Content-Type": "application/json",
    },
    data=json.dumps(
        {
            "model": "subq-snap-1.1",
            "state": (
                "Patient with asthma reports a sore throat, fever since "
                "yesterday, and shortness of breath walking to the kitchen "
                "this morning."
            ),
            "questions": {
                "escalate": {
                    "type": "noul",
                    "instructions": (
                        "Should a nurse see this chat before the patient "
                        "books anything?"
                    ),
                    "criteria": {
                        "true": (
                            "Breathing trouble, chest pain, or a flare of a "
                            "known condition"
                        ),
                        "false": (
                            "Stable symptoms the patient can book for "
                            "themselves"
                        ),
                    },
                },
            },
        }
    ).encode(),
)

try:
    with urllib.request.urlopen(request, timeout=60) as response:
        result = json.load(response)
except urllib.error.HTTPError as error:
    detail = json.load(error)["detail"]
    raise RuntimeError(
        f"{error.code} {detail['error_type']}: {detail['message']}"
    ) from error

print(f"Escalate: {result['answers']['escalate']['noul']:.2f}")

Escalate: 0.98


In [8]:
import os, time, httpx
from types import SimpleNamespace
from dotenv import load_dotenv

load_dotenv()

SUBQ_URL = "https://api.subq.ai/v1/systemone"
SUBQ_MODEL = "subq-snap-1.1"
TIMEOUT = httpx.Timeout(600.0, connect=30.0)
MAX_ITEMS = 128  # SubQ: "questions: must have at most 128 questions" pro Request (bei Drex: 512)

# Regeln einmal im State statt als Criteria pro Kandidat (wie bei Jev/Drex/Mammoth)
RULES = ("A column counts as referenced if it appears in SELECT, WHERE, a JOIN condition, GROUP BY, "
         "HAVING, ORDER BY, or inside an aggregation function of the SQL query that answers the question. "
         "If a table is used only via * or COUNT(*) and no other column of it is referenced, "
         "its first column counts as referenced.")


def query_subq(state, questions, retries=6):
    """Ein SubQ-Request. questions: {name: instruction}.
    Liefert ein Objekt mit .nouls[name].noul und .usage.input_tokens/.output_tokens
    (gleiche Form wie bei Jev/Drex/Mammoth, damit die Eval-Funktionen unveraendert funktionieren)."""
    payload = {
        "model": SUBQ_MODEL,
        "state": state,
        "questions": {name: {"type": "noul", "instructions": instr} for name, instr in questions.items()},
    }
    for attempt in range(retries):
        try:
            response = httpx.post(
                SUBQ_URL,
                headers={"Authorization": f"Bearer {os.environ['SUBQ_API_KEY']}"},
                json=payload,
                timeout=TIMEOUT,
            )
            if response.status_code >= 400:
                # Fehlerdetails von SubQ (z.B. validation_error) in die Exception uebernehmen
                raise httpx.HTTPStatusError(f"{response.status_code}: {response.text[:300]}",
                                            request=response.request, response=response)
            break
        except (httpx.HTTPStatusError, httpx.TransportError) as e:
            status = getattr(getattr(e, "response", None), "status_code", None)
            retryable = status is None or status == 429 or status >= 500
            if attempt == retries - 1 or not retryable:
                raise
            wait = 2 ** attempt
            if status == 429:
                wait = max(float(e.response.headers.get("retry-after", 0) or 0), 20)
            time.sleep(wait)
    body = response.json()
    return SimpleNamespace(
        nouls={k: SimpleNamespace(noul=a["noul"]) for k, a in body["answers"].items()},
        usage=SimpleNamespace(input_tokens=body["usage"]["input_tokens"],
                              output_tokens=body["usage"]["output_tokens"]),
    )

In [9]:
# Zweistufiges Schema Linking mit SubQ Snap (analog zu drex_max_test.ipynb):
#   Stufe 1: kompaktes Schema (Tabellen, Spalten, FK-Ziele) -> ein Noul pro Tabelle
#   Stufe 2: volle DDL nur der ausgewaehlten Tabellen -> ein Noul pro Spalte
# Gechunkt wird nicht nach Kontextfenster (~360k Tokens reichen), sondern nach der maximalen Fragenanzahl pro Request
# (MAX_ITEMS = 128, wie bei Drex mit 512).
import random

from src.utils import _extract_fk_targets, _pack_tables_into_chunks

TABLE_RULES = ("A table is needed if at least one of its columns is referenced in the SQL query that answers the question, "
               "or if it is needed to join other needed tables. "
               "The schema shown is only one part of a large database; the needed tables may be in another part.")


def compact_table(t):
    # 'tabelle(spalte1, spalte2) FK -> ziel1, ziel2' - ohne Typen, Backticks und Beispielwerte
    line = f"{t['candidates']['table']}({', '.join(t['candidates']['columns'])})"
    fks = _extract_fk_targets(t['ddl'])
    if fks:
        line += f" FK -> {', '.join(sorted(fks))}"
    return line


def chunk_compact(tables, max_tables):
    # Wie bei Drex: maximal max_tables Tabellen (= Fragen) pro Chunk, kompaktes Format
    chunks, current = [], []
    for t in tables:
        if current and len(current) >= max_tables:
            chunks.append(current)
            current = []
        current.append((t['candidates']['table'], compact_table(t)))
    if current:
        chunks.append(current)
    return chunks


def fk_neighbors(tables, selected):
    # Tabellen, die per FK direkt mit einer ausgewaehlten Tabelle verbunden sind (beide Richtungen)
    neighbors = set()
    for t in tables:
        name = t['candidates']['table'].lower()
        targets = {x.lower() for x in _extract_fk_targets(t['ddl'])}
        if name in selected:
            neighbors |= targets
        if targets & selected:
            neighbors.add(name)
    return neighbors - selected


def _ask(q, state, questions, responses):
    try:
        responses.append(query_subq(state=state, questions=questions))
    except Exception as e:
        print(f"Error processing question {q['question']} with db_id {q['db']}: {e}")


def merge_responses(responses):
    return SimpleNamespace(
        nouls={k: a for resp in responses for k, a in resp.nouls.items()},
        usage=SimpleNamespace(
            input_tokens=sum(resp.usage.input_tokens for resp in responses),
            output_tokens=sum(resp.usage.output_tokens for resp in responses),
        ),
    )


def run_subq_two_stage(data, table_threshold=0.3, include_fk_neighbors=False, max_items=MAX_ITEMS):
    results = []
    for q in data:
        t0 = time.perf_counter()

        # Stufe 1: Tabellen
        table_responses = []
        stage1_chunks = chunk_compact(q['db_schema'], max_items)
        for chunk in stage1_chunks:
            _ask(q,
                 state={"question": q['question'], "schema": "\n".join(line for _, line in chunk), "rules": TABLE_RULES},
                 questions={name: f"Is table {name} needed for the SQL query?" for name, _ in chunk},
                 responses=table_responses)
        table_scores = {k: a.noul for resp in table_responses for k, a in resp.nouls.items()}

        selected = {t.lower() for t, p in table_scores.items() if p >= table_threshold}
        if include_fk_neighbors:
            selected |= fk_neighbors(q['db_schema'], selected)
        selected_tables = [t for t in q['db_schema'] if t['candidates']['table'].lower() in selected]

        # Stufe 2: Spalten der ausgewaehlten Tabellen (ohne Kontextfenster, nur max_items Spalten pro Chunk)
        column_responses = []
        stage2_chunks = _pack_tables_into_chunks(selected_tables, q['question'], None, q['db'], None, None, max_items)
        for chunk in stage2_chunks:
            _ask(q,
                 state={"question": q['question'], "schema": chunk['ddl'], "rules": RULES},
                 questions={c: f"Is column {c} referenced in the SQL query?" for c in chunk['candidate_strings']},
                 responses=column_responses)

        results.append({"question": q['question'], "db_id": q['db'], "sql": q['sql'], "gold_schema": q['gold_schema'],
                        "table_scores": table_scores, "selected_tables": sorted(selected),
                        "n_chunks_stage1": len(stage1_chunks), "n_chunks_stage2": len(stage2_chunks),
                        "seconds": time.perf_counter() - t0,
                        "table_response": merge_responses(table_responses),
                        "response": merge_responses(column_responses)})
    return results


def stage1_table_recall(results, thresholds=(0.1, 0.2, 0.3, 0.4, 0.5)):
    # Recall auf Tabellenebene: Wie viele Gold-Tabellen haette Stufe 1 bei welcher Schwelle durchgelassen?
    print(f"{'thr':>5} {'R_tab':>7} {'R=1':>7} {'#tab':>7}")
    for thr in thresholds:
        hit = total = n_sel = all_found = 0
        for r in results:
            gold = {t.lower() for t in r['gold_schema'] if t}
            sel = {t.lower() for t, p in r['table_scores'].items() if p >= thr}
            hit += len(gold & sel)
            total += len(gold)
            n_sel += len(sel)
            all_found += gold <= sel
        print(f"{thr:>5.2f} {hit / total:>7.4f} {all_found / len(results):>7.4f} {n_sel / len(results):>7.2f}")


def print_timing(results):
    n = len(results)
    secs = sorted(r['seconds'] for r in results)
    print(f"Chunks/Frage: Stufe 1 = {sum(r['n_chunks_stage1'] for r in results) / n:.2f}, "
          f"Stufe 2 = {sum(r['n_chunks_stage2'] for r in results) / n:.2f}")
    print(f"Sekunden/Frage: Mittel = {sum(secs) / n:.2f}, Median = {secs[n // 2]:.2f}, max = {secs[-1]:.2f}")

In [10]:
# Eval: micro P/R/F6 ueber (table, column)-Paare, identisch zu jev_test.ipynb / drex_max_test.ipynb
import json


def response_to_dict(resp):
    return {
        "nouls": {k: a.noul for k, a in resp.nouls.items()},
        "usage": {"input_tokens": resp.usage.input_tokens,
                  "output_tokens": resp.usage.output_tokens},
    }


def save_results(results, file_name):
    serializable = []
    for r in results:
        item = {**r, "response": response_to_dict(r["response"])}
        if "table_response" in r:
            item["table_response"] = response_to_dict(r["table_response"])
        serializable.append(item)
    with open(f"{file_name}.json", "w", encoding="utf-8") as f:
        json.dump(serializable, f, indent=4)


def _gold_cols(gold):
    return {(t.lower(), c.lower()) for t, cols in gold.items() for c in (cols or []) if t and c}

def _scores(response):
    # 'table.column' -> noul-Wahrscheinlichkeit
    return {tuple(k.lower().split(".", 1)): a.noul for k, a in response.nouls.items()}

def eval_results(results, threshold=0.5, beta=6):
    tp = n_pred = n_gold = 0
    perfect_recall = 0
    for r in results:
        gold = _gold_cols(r["gold_schema"])
        pred = {k for k, p in _scores(r["response"]).items() if p >= threshold}
        tp += len(pred & gold)
        n_pred += len(pred)
        n_gold += len(gold)
        perfect_recall += gold <= pred
    p = tp / n_pred if n_pred else 0.0
    rec = tp / n_gold if n_gold else 0.0
    b2 = beta ** 2
    f = (1 + b2) * p * rec / (b2 * p + rec) if (b2 * p + rec) else 0.0
    return {"threshold": threshold, "precision": p, "recall": rec, f"f{beta}": f,
            "perfect_recall_rate": perfect_recall / len(results),
            "avg_pred_cols": n_pred / len(results), "tp": tp, "pred": n_pred, "gold": n_gold}

def present_results(results):
    print(f"Fragen: {len(results)}")

    # Gold-Spalten, die gar nicht unter den Kandidaten sind
    missing = sum(len(_gold_cols(r["gold_schema"]) - set(_scores(r["response"]))) for r in results)
    print(f"Gold-Spalten ohne Kandidat: {missing}")

    usage_in = sum(r["response"].usage.input_tokens for r in results)
    usage_out = sum(r["response"].usage.output_tokens for r in results)
    print(f"Tokens: input={usage_in:,}  output={usage_out:,}\n")

    print(f"{'thr':>5} {'P':>7} {'R':>7} {'F6':>7} {'R=1':>7} {'#pred':>7}")
    for thr in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        m = eval_results(results, thr)
        print(f"{thr:>5.2f} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f6']:>7.4f} "
              f"{m['perfect_recall_rate']:>7.4f} {m['avg_pred_cols']:>7.2f}")

    # Fehleranalyse bei 0.5: Fragen mit verpassten Gold-Spalten
    misses = []
    for r in results:
        s = _scores(r["response"])
        missed = {k: s.get(k) for k in _gold_cols(r["gold_schema"]) if s.get(k, 0) < 0.5}
        if missed:
            misses.append((r["question"], r["sql"], missed))
    print(f"\nFragen mit verpassten Gold-Spalten @0.5: {len(misses)}")
    for qtext, sql, missed in misses[:10]:
        print(f"- {qtext}\n  {sql}\n  missed: {missed}")

In [11]:
from src.data_loaders.spider_ent_data import get_spider_ent_data

# Gleiche 30 Fragen wie im Jev-, Drex- und Mammoth-Test (seed 42) -> direkt vergleichbar
spider_ent = get_spider_ent_data(max_items=MAX_ITEMS)
sample = random.Random(42).sample(spider_ent, 30)

results_two_stage = run_subq_two_stage(sample, table_threshold=0.3, include_fk_neighbors=False)

print("Stufe 1 (Tabellen):")
stage1_table_recall(results_two_stage)
print(f"Tokens Stufe 1: input={sum(r['table_response'].usage.input_tokens for r in results_two_stage):,}\n")

# 'Gold-Spalten ohne Kandidat' zaehlt hier die Gold-Spalten, deren Tabelle in Stufe 1 rausgefallen ist
print("Stufe 2 (Spalten):")
present_results(results_two_stage)
print()
print_timing(results_two_stage)

save_results(results_two_stage, "spider_ent_subq_two_stage_results_30_samples_seed_42")

Stufe 1 (Tabellen):
  thr   R_tab     R=1    #tab
 0.10  1.0000  1.0000   27.17
 0.20  1.0000  1.0000   12.30
 0.30  0.9762  0.9667    8.67
 0.40  0.9762  0.9667    7.00
 0.50  0.9762  0.9667    5.87
Tokens Stufe 1: input=1,219,654

Stufe 2 (Spalten):
Fragen: 30
Gold-Spalten ohne Kandidat: 1
Tokens: input=177,434  output=0

  thr       P       R      F6     R=1   #pred
 0.10  0.2319  0.9756  0.8978  0.9333   11.50
 0.20  0.3031  0.9390  0.8886  0.9000    8.47
 0.30  0.3439  0.9268  0.8862  0.9000    7.37
 0.40  0.3689  0.9268  0.8904  0.9000    6.87
 0.50  0.3968  0.9146  0.8835  0.8667    6.30
 0.60  0.4379  0.9024  0.8773  0.8333    5.63
 0.70  0.4867  0.8902  0.8707  0.8000    5.00
 0.80  0.5760  0.8780  0.8658  0.7667    4.17
 0.90  0.6569  0.8171  0.8117  0.7000    3.40

Fragen mit verpassten Gold-Spalten @0.5: 4
- Find the number of pets whose weight is heavier than 10.
  SELECT count(*) FROM pets WHERE weight  >  10
  missed: {('xelion33_petmgmt_pet_profile', 'pet_curr_wt_kg'): 

In [12]:
results_two_stage = run_subq_two_stage(spider_ent, table_threshold=0.3, include_fk_neighbors=False)

print("Stufe 1 (Tabellen):")
stage1_table_recall(results_two_stage)
print(f"Tokens Stufe 1: input={sum(r['table_response'].usage.input_tokens for r in results_two_stage):,}\n")

print("Stufe 2 (Spalten):")
present_results(results_two_stage)
print()
print_timing(results_two_stage)

save_results(results_two_stage, "spider_ent_subq_two_stage_results")

Stufe 1 (Tabellen):
  thr   R_tab     R=1    #tab
 0.10  0.9829  0.9751   27.96
 0.20  0.9737  0.9618   12.06
 0.30  0.9669  0.9518    7.93
 0.40  0.9600  0.9419    6.26
 0.50  0.9521  0.9302    5.26
Tokens Stufe 1: input=24,669,035

Stufe 2 (Spalten):
Fragen: 602
Gold-Spalten ohne Kandidat: 43
Tokens: input=3,129,683  output=0

  thr       P       R      F6     R=1   #pred
 0.10  0.2552  0.9642  0.8969  0.9369   10.17
 0.20  0.3514  0.9549  0.9126  0.9219    7.31
 0.30  0.4064  0.9469  0.9141  0.9103    6.27
 0.40  0.4467  0.9414  0.9140  0.8970    5.67
 0.50  0.4801  0.9302  0.9073  0.8804    5.21
 0.60  0.5138  0.9216  0.9022  0.8654    4.83
 0.70  0.5533  0.9105  0.8949  0.8405    4.43
 0.80  0.6056  0.8901  0.8790  0.7990    3.96
 0.90  0.6830  0.8537  0.8480  0.7475    3.36

Fragen mit verpassten Gold-Spalten @0.5: 72
- What are the names and release years for all the songs of the youngest singer?
  SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1
  missed: 